In [ ]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.observing import Observatory, Observation
from spacerocks.spice import SpiceKernel
from spacerocks.nbody import Simulation, Force
from spacerocks.orbfit import gauss, fit_orbit_lm
import numpy as np


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel()
kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load("/Users/kjnapier/data/spice/sb441-n16.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-1.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-2.bsp")
kernel.load("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/kjnapier/data/spice/latest_leapseconds.tls
Loading kernel: /Users/kjnapier/data/spice/sb441-n16.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-1.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-2.bsp
Loading kernel: /Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc


In [2]:
w84 = Observatory.from_obscode('w84')

In [3]:
epoch = Time.now()


rock = SpaceRock.from_horizons("arrokoth", epoch=epoch, origin="ssb", reference_plane="J2000")
sim = Simulation.horizons(epoch, "J2000", "ssb")
sim.add(rock)

In [4]:
observations = []    
for idx in range(0, 300, 30):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane="J2000", origin="ssb")
    rock = sim.get_particle("arrokoth")
    obs = rock.observe(observer)
    observations.append(obs)

In [5]:
smear = 0.2/3600 * (np.pi / 180)

In [6]:
smear

9.69627362219072e-07

In [7]:
simulated_observations = []
for obs in observations:
    ra = obs.ra
    dec = obs.dec
    ra += np.random.normal(0, smear)
    dec += np.random.normal(0, smear)
    epoch = obs.epoch
    observer = obs.observer
    cov = [[smear**2, 0], [0, smear**2]]
    simulated_o = Observation.from_astrometry(obs.epoch, ra, dec, obs.observer)
    simulated_o.set_covariance(cov)
    simulated_observations.append(simulated_o)

In [9]:
rocks = gauss(simulated_observations[0], simulated_observations[5], simulated_observations[9], min_distance=1e-6)

In [12]:
sim = Simulation.giants(rocks[0].epoch, "J2000", "ssb")

In [13]:
fit_orbit_lm(simulated_observations, rocks[0], sim)

Iteration: 0, chisq: 103.52196662402342, lambda: 0.01, ndof: 14
Iteration: 1, chisq: 15.343877302051236, lambda: 0.001, ndof: 14
Iteration: 2, chisq: 15.343877302051236, lambda: 0.01, ndof: 14
Iteration: 3, chisq: 15.343877302051236, lambda: 0.1, ndof: 14
Iteration: 4, chisq: 15.343877302051236, lambda: 1, ndof: 14
Iteration: 5, chisq: 15.343877302051236, lambda: 10, ndof: 14
Iteration: 6, chisq: 15.343877302051236, lambda: 100, ndof: 14
Iteration: 7, chisq: 15.343877302051236, lambda: 1000, ndof: 14
Iteration: 8, chisq: 15.343877302051236, lambda: 10000, ndof: 14
Iteration: 9, chisq: 15.343877302051236, lambda: 100000, ndof: 14
Iteration: 10, chisq: 15.343877302051236, lambda: 1000000, ndof: 14
Iteration: 11, chisq: 15.343877302051236, lambda: 10000000, ndof: 14
Iteration: 12, chisq: 15.343877302051236, lambda: 100000000, ndof: 14
Iteration: 13, chisq: 15.343877302051236, lambda: 1000000000, ndof: 14
Iteration: 14, chisq: 14.495234022419377, lambda: 100000000, ndof: 14
Iteration: 15, 

In [15]:
randj(simulated_observations, rocks[0], sim)

(VecStorage { data: [0.37171047506083643, 23.215107114409125, 28.081144684452802, 23.303708073722774, 11.773757388609706, 0.4001898567637691, 6.720831630716421, 6.262654473650706, 0.3991513412204933, 0.08880243171795763], nrows: Dyn(10), ncols: Const }, VecStorage { data: [-22543.675540986907, -22566.089701570036, -22750.437192797788, -22977.80785447401, -22721.164726391406, 23547.203918666826, 23319.148980061043, 23500.656687147624, -9923.322063409845, -23105.876609598454, -9868.294502235474, -10360.368559858556, -10452.93773991318, -9847.431121912108, -8143.421939799111, 10415.784601736088, 8395.462702281975, 8940.247849409388, 3930.7112824571445, -9725.226399888432, -3015.7786898521445, -2633.155931874853, -3119.603191592546, -4992.742666232175, -8599.07821944006, 3205.0094609936687, 7267.75779806843, 5497.314717484159, -21711.226070525445, -2950.905357006206, 3381810.398625545, 2708068.110931805, 2047582.610864751, 1378626.9631140158, 681544.6287419036, 136.62548101756045, 699717.3